[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

In [26]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [27]:

import torch
import torch.nn.functional as F

In [28]:
# ✏️ YOUR IMPLEMENTATION HERE

def linear_attention(Q, K, V):
    phiK = F.elu(K) + 1.0
    phiQ = F.elu(Q) + 1.0
    S = torch.einsum('hia,hib->hab', phiK,V)
    sum_phiK = torch.sum(phiK, dim=1)
    num = torch.einsum('hov,hvb->hob', phiQ, S)
    dr = torch.einsum('hov,hv->ho',phiQ, sum_phiK)
    attn = num / dr.unsqueeze(2)
    return attn
    # pass  # Replace this

In [29]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

Output shape: torch.Size([1, 8, 32])
Has NaN? False


In [30]:
from torch_judge import check
check('linear_attention')


🧪 Testing: Linear Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.2ms)
  ✅ [2/4] No NaN or Inf (2.8ms)
  ✅ [3/4] Gradient flow (1.8ms)
  ✅ [4/4] Runs fast on long sequences (linear complexity) (24.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (31.0ms total)
  Progress saved. Run status() to see your dashboard.



In [31]:
# ✏️ YOUR IMPLEMENTATION HERE

def linear_attention(Q, K, V):
    #causal
    phiK = F.elu(K) + 1.0
    phiQ = F.elu(Q) + 1.0
    S = torch.einsum('hia,hib->hiab', phiK,V)
    S = torch.cumsum(S,dim=1)
    sum_phiK = torch.cumsum(phiK, dim=1)
    num = torch.einsum('hov,hovb->hob', phiQ, S)
    dr = torch.einsum('hov,hov->ho',phiQ, sum_phiK)
    attn = num / dr.unsqueeze(-1)
    return attn
    # pass  # Replace this


In [32]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)              # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

# matches the naive quadratic form?
phiQ, phiK = F.elu(Q) + 1.0, F.elu(K) + 1.0
sim = (phiQ @ phiK.transpose(-2, -1)).tril()
ref = (sim @ V) / sim.sum(-1, keepdim=True)
print("Matches quadratic ref?", torch.allclose(out, ref, atol=1e-5))

# causal? perturbing token t must not move any output before t
K2, V2 = K.clone(), V.clone()
K2[:, 4] = 99.0; V2[:, 4] = 99.0
out2 = linear_attention(Q, K2, V2)
print("Causal?", torch.allclose(out[:, :4], out2[:, :4], atol=1e-6))
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))

Output shape: torch.Size([1, 8, 32])
Has NaN? False
Matches quadratic ref? True
Causal? True
Pos 0 == V[0]? True
